# Metal Spotlight Long-Conversation Constraint Retention With Talk2AI

This notebook uses the Apple Silicon `HookLLMMetal` Spotlight path to run a long-turn conversation experiment seeded from the Talk2AI dataset used in `tools/talk2ai_vllm_metal_long_conversation.ipynb`.

It compares ordinary generation against Spotlight-steered generation while a persistent constraint block is emphasized in the prompt and the Talk2AI conversation context grows. Each assistant reply is checked for state-of-being verb violations with a spaCy POS tagger.


### Setup

Run this notebook from the `vllm-metal` environment described in `notebooks/metal/README.md`. It assumes this repo is installed or importable in that environment.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "metal":
    repo_root = repo_root.parents[1]
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

plugin_src = repo_root / "vllm_hook_plugins"
if str(plugin_src) not in sys.path:
    sys.path.insert(0, str(plugin_src))

print(f"Repo root: {repo_root}")
print(f"Plugin source: {plugin_src}")

Repo root: /Users/timothyburley/opensource/vLLM-Hook
Plugin source: /Users/timothyburley/opensource/vLLM-Hook/vllm_hook_plugins


### Imports And Metal Environment

In [2]:
import json
import os
import time
import urllib.request
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from vllm import SamplingParams
from vllm_hook_plugins.metal import HookLLMMetal
from vllm_hook_plugins.utils.spotlight.utils import get_span_ranges
from transformers import AutoTokenizer

os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")
os.environ.setdefault("VLLM_METAL_USE_PAGED_ATTENTION", "0")
os.environ.setdefault("VLLM_METAL_MEMORY_FRACTION", "auto")

print("Metal Spotlight long-conversation environment configured")

INFO 09-10 07:22:40 [__init__.py:44] Available plugins for group vllm.platform_plugins:
INFO 09-10 07:22:40 [__init__.py:46] - metal -> vllm_metal:register
INFO 09-10 07:22:40 [__init__.py:49] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 09-10 07:22:44 [__init__.py:238] Platform plugin metal is activated
INFO 09-10 07:22:45 [importing.py:69] Triton not installed or not compatible; certain GPU-related functions will not be available.
Metal Spotlight long-conversation environment configured


### Load The Metal Spotlight Model

The default model is intentionally small for Apple Silicon iteration. Increase `max_model_len` and switch models after confirming the experiment works on your machine.

In [3]:
cache_dir = str(repo_root / "cache")
model = "Qwen/Qwen2-1.5B-Instruct"
MAX_MODEL_LEN = 32000


def build_metal_spotlight_llm():
    return HookLLMMetal(
        model=model,
        worker_name="probe_spotlight",
        download_dir=cache_dir,
        trust_remote_code=True,
        dtype=torch.float16,
        enable_hook=True,
        gpu_memory_utilization=0.30,
        max_model_len=MAX_MODEL_LEN,
        max_num_seqs=1,
        enforce_eager=True,
        enable_prefix_caching=False,
        enable_chunked_prefill=False,
    )


llm = build_metal_spotlight_llm()

SPOTLIGHT_TOKENIZER = AutoTokenizer.from_pretrained(
    model,
    cache_dir=cache_dir,
    trust_remote_code=True,
)
if SPOTLIGHT_TOKENIZER.pad_token is None:
    SPOTLIGHT_TOKENIZER.pad_token = SPOTLIGHT_TOKENIZER.eos_token

print(f"Model loaded: {model}")
print("Metal Spotlight worker enabled")
print("Spotlight span tokenizer ready")
print(f"Max model length: {MAX_MODEL_LEN}")


HookLLMMetal worker=probe_spotlight hooks_enabled=True
INFO 09-10 07:22:46 [utils.py:278] non-default args: {'trust_remote_code': True, 'download_dir': '/Users/timothyburley/opensource/vLLM-Hook/cache', 'dtype': torch.float16, 'max_model_len': 32000, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.3, 'max_num_batched_tokens': 32000, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': 'Qwen/Qwen2-1.5B-Instruct'}
WARNING 09-10 07:22:46 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_USE_V1
WARNING 09-10 07:22:46 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 09-10 07:22:46 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 09-10 07:22:47 [model.py:617] Resolved architecture: Qwen2ForCausalLM
WARNING 09-10 07:22:47 [model.py:2090]

mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

INFO 09-10 07:22:51 [model_lifecycle.py:187] Model loaded in 1.34s: Qwen/Qwen2-1.5B-Instruct
INFO 09-10 07:22:51 [cache_policy.py:680] MLX path: reporting 0.92 GB for scheduler admission control (one max-length sequence, max_model_len=32000)
INFO 09-10 07:22:51 [kv_cache_utils.py:1733] GPU KV cache size: 32,000 tokens
INFO 09-10 07:22:51 [kv_cache_utils.py:1734] Maximum concurrency for 32,000 tokens per request: 1.00x
INFO 09-10 07:22:51 [cache_policy.py:297] KV cache config received: 2000 blocks (MLX manages cache internally)
INFO 09-10 07:22:51 [model_runner.py:649] Warming up model...
INFO 09-10 07:22:51 [model_runner.py:655] Model warm-up complete
INFO 09-10 07:22:51 [core.py:302] init engine (profile, create kv cache, warmup model) took 0.09 s (compilation: 0.09 s)
Model loaded: Qwen/Qwen2-1.5B-Instruct
Metal Spotlight worker enabled
Spotlight span tokenizer ready
Max model length: 32000


### Load Talk2AI Conversations

Point `TALK2AI_LOCAL_PATH` at the real Talk2AI JSON when you have it. The loader matches the original long-conversation notebook: it tries a local file, then the paper-indicated GitHub URL, then falls back to a small synthetic sample with the same shape so the notebook remains runnable.


In [4]:
# If you already have a copy of talk2ai_conversations.json, point this at it.
TALK2AI_LOCAL_PATH = None  # e.g. "/Users/you/Downloads/talk2ai_conversations.json"

TALK2AI_CANDIDATE_URLS = [
    "https://raw.githubusercontent.com/MassimoStel/Talk2AI/main/Data_paper/talk2ai_conversations.json",
]


def _download_json(url, timeout=15):
    req = urllib.request.Request(url, headers={"User-Agent": "vllm-hook-talk2ai-notebook/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def build_synthetic_talk2ai_sample():
    return [
        {
            "conversationId": "synthetic-climate-001",
            "topic": "climate change",
            "messages": [
                {"role": "user", "content": "Honestly, I think how much climate change gets talked about is overblown compared to other problems we have."},
                {"role": "assistant", "content": "That's a common view, so let's look at it together. What makes it feel overblown to you - the media coverage, the urgency of the language, or something else?"},
                {"role": "user", "content": "Mostly the media coverage. It feels like every week there's a new 'worst ever' headline."},
                {"role": "assistant", "content": "That pattern is real, and it can create headline fatigue even when the underlying trend is genuinely worsening. Would it help to look at a couple of long-run measurements instead of headlines?"},
            ],
        },
        {
            "conversationId": "synthetic-mathanxiety-001",
            "topic": "math anxiety",
            "messages": [
                {"role": "user", "content": "I freeze up during math tests even when I know the material. It's embarrassing."},
                {"role": "assistant", "content": "That's a really common experience and it has a name - math anxiety - and it's not about ability. Does the freezing happen mostly during timed tests, or also with untimed homework?"},
                {"role": "user", "content": "Mostly timed tests. My mind just goes blank."},
                {"role": "assistant", "content": "Time pressure is one of the biggest triggers for that blanking response. A few students find it helps to do a short breathing reset before the clock starts - want to walk through one?"},
            ],
        },
        {
            "conversationId": "synthetic-healthmisinfo-001",
            "topic": "health misinformation",
            "messages": [
                {"role": "user", "content": "A friend sent me a post saying a common vaccine ingredient causes serious harm. Is that true?"},
                {"role": "assistant", "content": "That specific claim has been examined closely and hasn't held up under review. Do you want me to walk through what the ingredient actually does and where that claim seems to have come from?"},
                {"role": "user", "content": "Yeah, I'd like to understand where it came from."},
                {"role": "assistant", "content": "Sure - a lot of claims like this trace back to a single misinterpreted or retracted study that keeps getting recirculated. Let's look at the original source."},
            ],
        },
    ]


def load_talk2ai(local_path=TALK2AI_LOCAL_PATH, candidate_urls=TALK2AI_CANDIDATE_URLS):
    if local_path and os.path.exists(local_path):
        print(f"Loading Talk2AI conversations from local file: {local_path}")
        with open(local_path) as f:
            return json.load(f), "local_file"

    for url in candidate_urls:
        try:
            print(f"Attempting to download Talk2AI conversations from {url} ...")
            data = _download_json(url)
            print("Download succeeded.")
            return data, "remote_download"
        except Exception as exc:
            print(f"  could not fetch from {url}: {exc}")

    print("Could not locate the real Talk2AI dataset. Falling back to the synthetic Talk2AI-shaped sample.")
    return build_synthetic_talk2ai_sample(), "synthetic_fallback"


conversations, dataset_source = load_talk2ai()
print(f"Loaded {len(conversations)} conversation(s) - source: {dataset_source}")


Attempting to download Talk2AI conversations from https://raw.githubusercontent.com/MassimoStel/Talk2AI/main/Data_paper/talk2ai_conversations.json ...
  could not fetch from https://raw.githubusercontent.com/MassimoStel/Talk2AI/main/Data_paper/talk2ai_conversations.json: HTTP Error 404: Not Found
Could not locate the real Talk2AI dataset. Falling back to the synthetic Talk2AI-shaped sample.
Loaded 3 conversation(s) - source: synthetic_fallback


### Select A Talk2AI Seed Conversation

The experiment replays user turns from this Talk2AI seed and inserts newly generated assistant turns for each condition. Change `SEED_INDEX` or filter by `topic` to try a different conversation.


In [5]:
SEED_INDEX = 0
seed = conversations[SEED_INDEX]
seed_messages = [m for m in seed.get("messages", []) if m.get("role") in {"user", "assistant"}]
USER_TURNS = [m["content"] for m in seed_messages if m.get("role") == "user"]

if not USER_TURNS:
    raise ValueError("Selected Talk2AI conversation has no user turns to replay.")

print("conversationId:", seed.get("conversationId"))
print("topic         :", seed.get("topic"))
print("dataset_source:", dataset_source)
print("seed user turns:", len(USER_TURNS))
print()
for m in seed_messages[:8]:
    print(f"[{m['role']}] {m['content']}")


conversationId: synthetic-climate-001
topic         : climate change
dataset_source: synthetic_fallback
seed user turns: 2

[user] Honestly, I think how much climate change gets talked about is overblown compared to other problems we have.
[assistant] That's a common view, so let's look at it together. What makes it feel overblown to you - the media coverage, the urgency of the language, or something else?
[user] Mostly the media coverage. It feels like every week there's a new 'worst ever' headline.
[assistant] That pattern is real, and it can create headline fatigue even when the underlying trend is genuinely worsening. Would it help to look at a couple of long-run measurements instead of headlines?


### Experiment Configuration

Each turn asks the model to answer a fresh user message while preserving the same constraints. Spotlight emphasizes the constraint block in the full prompt.

In [6]:
CONSTRAINT_BLOCK = """Persistent constraints:
1. Do not use state of being verbs [am, is, are, was, were, be, being, been], 
or contractions containing state of being verbs.
""".strip()


SYSTEM_MESSAGE = (
    "You are a concise assistant continuing a Talk2AI-style conversation about "
    f"{seed.get('topic', 'the selected topic')}. Preserve nuance, acknowledge uncertainty, "
    "and adapt as the participant's concerns evolve."
)
ALPHA = 0.2
SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=96)
TARGET_USER_TURNS = 50
HISTORY_WINDOW_MESSAGES = 12  # six recent user/assistant exchanges kept in each prompt

CONVERSATION_ARCS = {
    "climate change": [
        "I get that headlines can exaggerate, but I still wonder whether the long-term measurements are being framed fairly.",
        "If the temperature trend is real, why do people in my town still argue from whatever the weather was like this winter?",
        "I am not denying the data, but I worry that every proposed fix seems expensive for ordinary households.",
        "What is the strongest argument that current climate policy asks too much too quickly?",
        "How should I think about countries that industrialized earlier asking poorer countries to slow down now?",
        "A friend says adaptation is more realistic than prevention. Where is that view right, and where does it break down?",
        "Can you separate personal choices, local policy, and national policy without making one sound like the whole answer?",
        "I hear people say the models have been wrong before. What kind of uncertainty actually matters here?",
        "If someone works in oil, gas, shipping, or farming, what would a fair transition even mean?",
        "I am bothered by activists who sound certain about everything. Does that hurt the credibility of the underlying science?",
        "What would be a practical climate action that is boring but actually high leverage?",
        "How do we avoid climate concern turning into a status signal instead of useful behavior?",
        "If electricity demand is growing because of AI and data centers, does that change the clean energy argument?",
        "What is a realistic timeline for change that is urgent but not fantasy?",
        "Can you explain how risk compounds without making it sound like immediate doom?",
        "Where do carbon offsets fit in, if at all? I hear both praise and accusations of greenwashing.",
        "What would you say to someone who thinks climate policy is mostly political control?",
        "How do insurance prices, food prices, and infrastructure costs make this issue less abstract?",
        "I want to talk about tradeoffs honestly. What gets worse if we move too slowly, and what gets worse if we move too fast?",
        "What is the best local question to ask a city council candidate about this?",
        "Suppose I only changed one habit this year. Which change would be meaningful without pretending I solved the problem?",
        "How do I tell the difference between serious criticism of a policy and misinformation about the science?",
        "What is one thing climate communicators should stop doing?",
        "What is one thing skeptics should concede if they are arguing in good faith?",
        "Can you end this part by giving me a position that is cautious, practical, and still evidence-based?",
    ],
    "math anxiety": [
        "I know practice matters, but timed tests make me feel like my brain is shutting down before I start.",
        "How can I tell the difference between not understanding the material and panicking while I do understand it?",
        "I hate advice that sounds like just relax. What is a concrete first step during the first minute of a test?",
        "What should a teacher change that would help anxious students without lowering standards?",
        "If I use accommodations, I worry people will think I am getting an unfair advantage. How should I think about that?",
        "Can you explain why smart people can still freeze on basic problems?",
        "What is a good way to review mistakes without turning it into proof that I am bad at math?",
        "How should parents talk about grades when anxiety is part of the problem?",
        "What is the role of memorization versus conceptual understanding here?",
        "Can you give an example of a study routine that builds confidence gradually?",
        "What if the anxiety comes from one bad teacher or one humiliating class experience?",
        "How do I ask for help without sounding like I am making excuses?",
        "What would progress look like over a month, not overnight?",
        "Are there cases where pushing through is helpful and cases where it backfires?",
        "How should I handle a blank mind when everyone else seems to be working?",
        "What can a school do to avoid making math identity feel fixed?",
        "Can you compare test anxiety in math with stage fright or sports pressure?",
        "What should I do after a bad score so I do not spiral?",
        "How do I keep ambition without tying my self-worth to performance?",
        "What is a fair way to measure improvement when the grade is still uneven?",
        "Can you end this part with a plan that is compassionate but not vague?",
    ],
    "health misinformation": [
        "I do not want to overreact to a viral post, but I also do not want to dismiss a real safety concern.",
        "How can I check a health claim without needing a medical degree?",
        "What makes a source trustworthy besides having a professional-looking website?",
        "If someone had a bad experience after a treatment, how do we respect that without assuming causation?",
        "Why do ingredient names sound scary even when the dose or context matters?",
        "What should I say to a friend who thinks fact-checking sites are biased?",
        "Can you explain absolute risk versus relative risk with a health example?",
        "What is the strongest fair criticism of public health messaging in recent years?",
        "How do I talk to family without making them defensive?",
        "What warning signs suggest a post is selling fear rather than information?",
        "If official guidance changes, how do I know whether that is learning or incompetence?",
        "What does consensus mean if individual doctors sometimes disagree?",
        "How should I think about rare side effects without exaggerating or ignoring them?",
        "What is a practical checklist before sharing a health claim?",
        "How do influencers exploit uncertainty in medical topics?",
        "What would make you pause and investigate a claim further?",
        "Can you explain why anecdotes feel more persuasive than population data?",
        "What is one respectful question I can ask someone who believes a dubious claim?",
        "How do platforms reward the most alarming version of a health story?",
        "Can you end this part with a cautious, humane approach to evaluating claims?",
    ],
}

GENERAL_ARC = [
    "I see your point, but I still feel conflicted because the issue affects different groups unevenly.",
    "Can you separate what is strongly supported from what is still uncertain? I do not want a one-sided answer.",
    "What would someone skeptical of your last point say, and which part of that criticism is fair?",
    "Bring this back to everyday decisions. What would change for a person who only has limited time and money?",
    "Where do values enter the discussion, not just facts?",
    "What evidence would actually change your mind about this?",
    "Can you distinguish between a good-faith concern and a misleading talking point?",
    "What should I watch for over the next few years to update my view?",
]

PHASE_BRIDGES = [
    "Let me push on that from a practical angle.",
    "I want to make this less abstract.",
    "Here is the part I still find hard to reconcile.",
    "Suppose I am talking to someone who disagrees with me.",
    "I am trying to update my view without just adopting a slogan.",
    "Before moving on, I want to test the tradeoff.",
]


def topic_arc(topic):
    normalized = (topic or "").lower()
    for key, turns in CONVERSATION_ARCS.items():
        if key in normalized:
            return turns
    return GENERAL_ARC


def build_long_user_turns(seed_turns, topic, target_turns):
    turns = [turn.strip() for turn in seed_turns if turn and turn.strip()]
    if not turns:
        turns = [f"I want to talk through {topic}, but I am not sure what to believe yet."]

    arc = topic_arc(topic)
    index = 0
    while len(turns) < target_turns:
        bridge = PHASE_BRIDGES[index % len(PHASE_BRIDGES)]
        angle = arc[index % len(arc)]
        if index >= len(arc):
            angle = f"Returning to this from another angle: {angle}"
        turns.append(f"{bridge} {angle}")
        index += 1
    return turns[:target_turns]


TALK2AI_USER_TURNS = list(USER_TURNS)
USER_TURNS = build_long_user_turns(
    TALK2AI_USER_TURNS,
    seed.get("topic", "the selected Talk2AI topic"),
    TARGET_USER_TURNS,
)
MAX_USER_TURNS = len(USER_TURNS)

print(CONSTRAINT_BLOCK)
print(f"Seed Talk2AI user turns: {len(TALK2AI_USER_TURNS)}")
print(f"Expanded user turns: {MAX_USER_TURNS}")
print(f"Prompt history window messages: {HISTORY_WINDOW_MESSAGES}")
print("Preview of final generated user turn:")
print(USER_TURNS[-1])


Persistent constraints:
1. Do not use state of being verbs [am, is, are, was, were, be, being, been], 
or contractions containing state of being verbs.
Seed Talk2AI user turns: 2
Expanded user turns: 50
Prompt history window messages: 12
Preview of final generated user turn:
Before moving on, I want to test the tradeoff. Returning to this from another angle: What is one thing climate communicators should stop doing?


### Prompt And Scoring Helpers

In [7]:
def render_prompt(history, user_message):
    recent_history = history[-HISTORY_WINDOW_MESSAGES:]
    omitted_messages = max(0, len(history) - len(recent_history))
    transcript = "\n".join(
        f"{item['role'].upper()}: {item['content']}" for item in recent_history
    )
    if transcript:
        transcript = transcript + "\n"

    earlier_context = ""
    if omitted_messages:
        earlier_context = (
            f"Earlier conversation context: {omitted_messages} older messages are omitted "
            f"from this prompt to keep the Metal run within memory limits. Continue the same "
            f"Talk2AI conversation about {seed.get('topic')}.\n\n"
        )

    return f"""{SYSTEM_MESSAGE}

Talk2AI conversation id: {seed.get('conversationId')}
Topic: {seed.get('topic')}

{CONSTRAINT_BLOCK}

{earlier_context}Recent conversation:
{transcript}USER: {user_message}
ASSISTANT:""".strip()


def clean_text(text):
    return text.strip().split("\n\n")[0].strip()


def build_spotlight_sampling_params(prompt, base_params, emph_string, alpha):
    tokenized = SPOTLIGHT_TOKENIZER(
        [prompt],
        return_tensors="pt",
        return_offsets_mapping=True,
        padding=True,
    )
    offset_mappings = tokenized.pop("offset_mapping")
    span_ranges = get_span_ranges([prompt], [[emph_string]], offset_mappings)

    import copy

    params = copy.copy(base_params)
    extra = dict(params.extra_args or {})
    extra["spotlight"] = {
        "span_ranges": span_ranges[0],
        "alpha": alpha,
    }
    params.extra_args = extra
    return params


### Run Baseline And Spotlight Conditions

In [8]:
def run_condition(name, use_spotlight, model_runner):
    history = []
    rows = []
    for turn_index, user_message in enumerate(USER_TURNS[:MAX_USER_TURNS], start=1):
        prompt = render_prompt(history, user_message)
        start = time.perf_counter()
        if use_spotlight:
            spotlight_params = build_spotlight_sampling_params(
                prompt,
                SAMPLING_PARAMS,
                CONSTRAINT_BLOCK,
                ALPHA,
            )
            outputs = model_runner.generate(
                prompts=[prompt],
                sampling_params=spotlight_params,
                use_hook=True,
            )
        else:
            outputs = model_runner.generate(
                prompts=[prompt],
                sampling_params=SAMPLING_PARAMS,
                use_hook=False,
            )
        elapsed_s = time.perf_counter() - start
        reply = clean_text(outputs[0].outputs[0].text)
        rows.append(
            {
                "condition": name,
                "turn": turn_index,
                "conversation_id": seed.get("conversationId"),
                "topic": seed.get("topic"),
                "dataset_source": dataset_source,
                "user_message": user_message,
                "prompt_chars": len(prompt),
                "history_messages_in_prompt": min(len(history), HISTORY_WINDOW_MESSAGES),
                "latency_s": elapsed_s,
                "reply": reply,
            }
        )
        history.extend(
            [
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": reply},
            ]
        )
        print(f"{name} turn {turn_index}: latency={elapsed_s:.2f}s")
    return rows


baseline_rows = run_condition("baseline", use_spotlight=False, model_runner=llm)

# Free the baseline engine before the Spotlight condition. HookLLMMetal's Spotlight
# path builds a hook-enabled engine per call, and the memory guard checks available
# host memory before that setup starts.
llm.shutdown()
del llm
import gc
gc.collect()

llm = build_metal_spotlight_llm()
spotlight_rows = run_condition("spotlight", use_spotlight=True, model_runner=llm)
results = pd.DataFrame(baseline_rows + spotlight_rows)
results


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.53s/it, est. speed input: 34.27 toks/s, output: 27.19 toks/s]

baseline turn 1: latency=3.56s


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.43s/it, est. speed input: 65.33 toks/s, output: 28.00 toks/s]

...

INFO 09-10 07:30:22 [model.py:617] Resolved architecture: Qwen2ForCausalLM
WARNING 09-10 07:30:22 [model.py:2090] Casting torch.bfloat16 to torch.float16.
INFO 09-10 07:30:22 [model.py:1752] Using max model len 32000
WARNING 09-10 07:30:22 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-10 07:30:22 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 09-10 07:30:22 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 09-10 07:30:22 [platform.py:324] Metal memory: 34.4GB total, 12.2GB available
INFO 09-10 07:30:23 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='Qwen/Qwen2-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2-1.5B-Instruct', skip_tokeniz

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.71s/it, est. speed input: 270.19 toks/s, output: 25.89 toks/s]

Metal steer: hooked generation complete
Metal steer: cleaning up hook run
Metal steer: host after hook exit rss=9.92 GB available=9.79 GB total=32.00 GB
spotlight turn 50: latency=5.65s


,condition,turn,conversation_id,topic,dataset_source,user_message,prompt_chars,history_messages_in_prompt,latency_s,reply
0,baseline,1,synthetic-climate-001,climate change,synthetic_fallback,"Honestly, I think how much climate change gets...",550,0,3.555923,That's a valid concern. Climate change is a co...
1,baseline,2,synthetic-climate-001,climate change,synthetic_fallback,Mostly the media coverage. It feels like every...,1061,2,3.438996,"Yes, media coverage can certainly influence pu..."
2,baseline,3,synthetic-climate-001,climate change,synthetic_fallback,Let me push on that from a practical angle. I ...,1743,4,3.583803,That's a valid point. It's important to consid...
3,baseline,4,synthetic-climate-001,climate change,synthetic_fallback,I want to make this less abstract. If the temp...,2410,6,3.400764,That's a good point. It's important to remembe...
4,baseline,5,synthetic-climate-001,climate change,synthetic_fallback,Here is the part I still find hard to reconcil...,3048,8,3.664230,That's a valid concern. Climate change mitigat...
...,...,...,...,...,...,...,...,...,...,...
95,spotlight,46,synthetic-climate-001,climate change,synthetic_fallback,I want to make this less abstract. Returning t...,5023,12,6.474335,That's a valid point. Climate change mitigatio...
96,spotlight,47,synthetic-climate-001,climate change,synthetic_fallback,Here is the part I still find hard to reconcil...,5004,12,5.896312,That's a valid point. Climate change mitigatio...
97,spotlight,48,synthetic-climate-001,climate change,synthetic_fallback,Suppose I am talking to someone who disagrees ...,5056,12,5.192284,That's a valid point. Climate change mitigatio...
98,spotlight,49,synthetic-climate-001,climate change,synthetic_fallback,I am trying to update my view without just ado...,5088,12,5.438659,That's a valid point. Climate change mitigatio...


### Check Constraint Retention

The active persistent constraint is avoidance of state-of-being verbs. This checker uses spaCy's POS tagger and lemmatizer so contractions such as `I'm`, `it's`, `you're`, and `we've been` are detected through their parsed `be` tokens instead of only by surface string matching. It annotates the generated replies before the CSV is saved, and the same helper can score a previously saved CSV.


In [9]:
STATE_OF_BEING_CORE_WORDS = {"am", "is", "are", "was", "were", "be", "being", "been"}
STATE_OF_BEING_SPACY_MODEL = "en_core_web_sm"


def load_state_of_being_nlp(model_name=STATE_OF_BEING_SPACY_MODEL):
    try:
        import spacy
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "spaCy is required for the state-of-being checker. Install it with "
            "`python -m pip install spacy`, then install the English model with "
            "`python -m spacy download en_core_web_sm`."
        ) from exc

    try:
        return spacy.load(model_name, disable=["ner", "parser"])
    except OSError as exc:
        raise OSError(
            f"spaCy model {model_name!r} is not installed. Run "
            f"`python -m spacy download {model_name}` before executing this cell."
        ) from exc


state_of_being_nlp = load_state_of_being_nlp()


def is_state_of_being_token(token):
    lemma = token.lemma_.lower()
    text = token.text.lower()

    # Main path: spaCy splits contractions and lemmatizes them to `be`, e.g. `I'm` -> 'm/AUX/be.
    if lemma == "be" and token.pos_ in {"AUX", "VERB"}:
        return True

    # Conservative fallback for model/tagger edge cases on the eight explicit surface forms.
    return text in STATE_OF_BEING_CORE_WORDS and token.pos_ in {"AUX", "VERB"}


def check_state_of_being_verbs(text, nlp=state_of_being_nlp):
    doc = nlp(text or "")
    matches = []
    for token in doc:
        if is_state_of_being_token(token):
            matches.append(
                {
                    "text": token.text,
                    "lemma": token.lemma_,
                    "pos": token.pos_,
                    "tag": token.tag_,
                    "start": token.idx,
                    "end": token.idx + len(token.text),
                }
            )
    return {
        "state_of_being_count": len(matches),
        "no_state_of_being_verbs": len(matches) == 0,
        "state_of_being_score": 1.0 if len(matches) == 0 else 0.0,
        "state_of_being_matches": matches,
    }


def add_state_of_being_scores(frame, text_column="reply", nlp=state_of_being_nlp):
    scored = frame.copy()
    checks = [check_state_of_being_verbs(text, nlp=nlp) for text in scored[text_column].fillna("")]
    scored["state_of_being_count"] = [check["state_of_being_count"] for check in checks]
    scored["no_state_of_being_verbs"] = [check["no_state_of_being_verbs"] for check in checks]
    scored["state_of_being_score"] = [check["state_of_being_score"] for check in checks]
    scored["state_of_being_matches"] = [
        ", ".join(match["text"] for match in check["state_of_being_matches"])
        for check in checks
    ]
    return scored


def score_state_of_being_csv(csv_file, text_column="reply", nlp=state_of_being_nlp):
    frame = pd.read_csv(csv_file)
    return add_state_of_being_scores(frame, text_column=text_column, nlp=nlp)


results = add_state_of_being_scores(results)
results["retention_score"] = results["state_of_being_score"]
results[["condition", "turn", "retention_score", "state_of_being_count", "state_of_being_matches", "reply"]].head()


,condition,turn,retention_score,state_of_being_count,state_of_being_matches,reply
0,baseline,1,0.0,6,"'s, is, 's, 's, is, 's",That's a valid concern. Climate change is a co...
1,baseline,2,0.0,6,"'s, is, 's, is, 's, 're","Yes, media coverage can certainly influence pu..."
2,baseline,3,0.0,5,"'s, 's, 's, is, 's",That's a valid point. It's important to consid...
3,baseline,4,0.0,9,"'s, 's, is, 's, 's, 's, is, 's, is",That's a good point. It's important to remembe...
4,baseline,5,0.0,8,"'s, be, 's, 's, is, 's, 's, are",That's a valid concern. Climate change mitigat...


### Compare Constraint Retention

In [10]:
summary = results.groupby("condition").agg(
    mean_retention=("retention_score", "mean"),
    state_of_being_retention=("state_of_being_score", "mean"),
    state_of_being_violation_rate=("no_state_of_being_verbs", lambda values: 1 - values.mean()),
    mean_state_of_being_count=("state_of_being_count", "mean"),
    mean_latency_s=("latency_s", "mean"),
    max_prompt_chars=("prompt_chars", "max"),
)
summary


,mean_retention,state_of_being_retention,state_of_being_violation_rate,mean_state_of_being_count,mean_latency_s,max_prompt_chars
condition,,,,,,
baseline,0.0,0.0,1.0,7.9,3.700885,5132
spotlight,0.0,0.0,1.0,7.9,5.349533,5132


### Inspect Failures

In [11]:
failures = results[results["retention_score"] < 1.0][
    ["condition", "turn", "user_message", "retention_score", "state_of_being_count", "state_of_being_matches", "reply"]
]
failures


,condition,turn,user_message,retention_score,state_of_being_count,state_of_being_matches,reply
0,baseline,1,"Honestly, I think how much climate change gets...",0.0,6,"'s, is, 's, 's, is, 's",That's a valid concern. Climate change is a co...
1,baseline,2,Mostly the media coverage. It feels like every...,0.0,6,"'s, is, 's, is, 's, 're","Yes, media coverage can certainly influence pu..."
2,baseline,3,Let me push on that from a practical angle. I ...,0.0,5,"'s, 's, 's, is, 's",That's a valid point. It's important to consid...
3,baseline,4,I want to make this less abstract. If the temp...,0.0,9,"'s, 's, is, 's, 's, 's, is, 's, is",That's a good point. It's important to remembe...
4,baseline,5,Here is the part I still find hard to reconcil...,0.0,8,"'s, be, 's, 's, is, 's, 's, are",That's a valid concern. Climate change mitigat...
...,...,...,...,...,...,...,...
95,spotlight,46,I want to make this less abstract. Returning t...,0.0,8,"'s, be, 's, 's, is, 's, 's, are",That's a valid point. Climate change mitigatio...
96,spotlight,47,Here is the part I still find hard to reconcil...,0.0,8,"'s, be, 's, 's, is, 's, 's, are",That's a valid point. Climate change mitigatio...
97,spotlight,48,Suppose I am talking to someone who disagrees ...,0.0,8,"'s, be, 's, 's, is, 's, 's, are",That's a valid point. Climate change mitigatio...
98,spotlight,49,I am trying to update my view without just ado...,0.0,8,"'s, be, 's, 's, is, 's, 's, are",That's a valid point. Climate change mitigatio...


### Save Results

In [12]:
out_dir = repo_root / "notebooks" / "metal" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
csv_path = out_dir / f"spotlight_state_of_being_retention_{stamp}.csv"
json_path = out_dir / f"spotlight_state_of_being_retention_{stamp}.json"

results.to_csv(csv_path, index=False)
json_path.write_text(
    json.dumps(
        {
            "model": model,
            "dataset_source": dataset_source,
            "source_conversation_id": seed.get("conversationId"),
            "topic": seed.get("topic"),
            "alpha": ALPHA,
            "constraint_block": CONSTRAINT_BLOCK,
            "retention_metric": "no_state_of_being_verbs",
            "target_user_turns": TARGET_USER_TURNS,
            "history_window_messages": HISTORY_WINDOW_MESSAGES,
            "max_model_len": MAX_MODEL_LEN,
            "seed_user_turns": len(TALK2AI_USER_TURNS),
            "summary": summary.reset_index().to_dict(orient="records"),
            "rows": results.to_dict(orient="records"),
        },
        indent=2,
    )
)

print(f"Saved CSV: {csv_path}")
print(f"Saved JSON: {json_path}")


Saved CSV: /Users/timothyburley/opensource/vLLM-Hook/notebooks/metal/results/spotlight_state_of_being_retention_20260910-073029.csv
Saved JSON: /Users/timothyburley/opensource/vLLM-Hook/notebooks/metal/results/spotlight_state_of_being_retention_20260910-073029.json
